# VERITAS AI: Forensic Exploratory Data Analysis & Feature Engineering
### Intelligent Image Forgery Detection and Digital Forensics System
**Author:** Akash Mishra | **Project:** B.Tech CSE (AI & ML) Capstone Project

---
### Abstract
This notebook demonstrates the end-to-end multi-domain forensic analysis pipeline:
1. **Error Level Analysis (ELA)**: Compression rate variance and JPEG quantization artifacts.
2. **Photo-Response Non-Uniformity (PRNU) & Sensor Noise**: High-pass Laplacian noise residuals.
3. **Copy-Move (Cloning) Keypoint Matching**: SIFT/ORB keypoints and translation vector clustering.
4. **20-Dimensional Feature Extraction**: Statistical moments, edge density, chromatic correlation, and 2D DCT.
5. **Statistical Distribution Comparison**: Authentic vs. Manipulated image distributions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import validate_image_bytes, load_image_rgb
from src.forensics.ela import perform_ela
from src.forensics.noise_analysis import analyze_noise_inconsistency
from src.forensics.copy_move import detect_copy_move
from src.features.extractor import extract_forensic_features, FEATURE_NAMES
from src.ml.classifier import VeritasClassifier

print('VERITAS AI Forensic Modules Imported Successfully.')

## 1. Load and Inspect Forensic Sample Images
We evaluate an authentic photograph and compare it directly against a spliced image and a copy-move tampered image.

In [ ]:
test_dir = os.path.join(PROJECT_ROOT, 'data', 'sample_images', 'test_samples')
auth_path = os.path.join(test_dir, 'sample_authentic_nature.jpg')
spliced_path = os.path.join(test_dir, 'sample_spliced_object.jpg')
clone_path = os.path.join(test_dir, 'sample_copy_move_cloned.jpg')

img_auth = Image.open(auth_path).convert('RGB')
img_spliced = Image.open(spliced_path).convert('RGB')
img_clone = Image.open(clone_path).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_auth)
axes[0].set_title('Authentic Image (Uniform Q=95)')
axes[0].axis('off')

axes[1].imshow(img_spliced)
axes[1].set_title('Spliced Image (Foreign Insert Q=65)')
axes[1].axis('off')

axes[2].imshow(img_clone)
axes[2].set_title('Copy-Move Image (Cloned Patch)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2. Error Level Analysis (ELA) Comparison
Notice how in the authentic image, error levels are uniformly distributed across homogeneous areas, while in the spliced image, the spliced donor region exhibits sharp error contrast due to inconsistent JPEG compression generations.

In [ ]:
ela_auth = perform_ela(img_auth, quality=90, scale_factor=20.0)
ela_spliced = perform_ela(img_spliced, quality=90, scale_factor=20.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(ela_auth.ela_image)
axes[0].set_title(f'Authentic ELA (Mean Error: {ela_auth.mean_error:.2f}, Std: {ela_auth.std_error:.2f})')
axes[0].axis('off')

axes[1].imshow(ela_spliced.ela_image)
axes[1].set_title(f'Spliced ELA (Mean Error: {ela_spliced.mean_error:.2f}, Std: {ela_spliced.std_error:.2f})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Copy-Move Keypoint Matching & Spatial Clustering

In [ ]:
cm_res = detect_copy_move(img_clone)
print(f'Keypoints Detected: {cm_res.keypoints_detected}')
print(f'Matched Pairs: {cm_res.matched_pairs_count}')
print(f'Translation Clusters: {cm_res.clustered_clusters_count}')
print(f'Copy-Move Detected Flag: {cm_res.copy_move_detected}')
print(f'Interpretation: {cm_res.forensic_interpretation}')

plt.figure(figsize=(8, 8))
plt.imshow(cm_res.visualization_image)
plt.title('Detected Copy-Move Cloned Keypoints & Translation Vectors')
plt.axis('off')
plt.show()

## 4. Multi-Domain Feature Vector Inspection & Dataset Distribution

In [ ]:
features_csv = os.path.join(PROJECT_ROOT, 'data', 'extracted_features.csv')
if os.path.exists(features_csv):
    df_feats = pd.read_csv(features_csv)
    print(f'Loaded {len(df_feats)} dataset feature vectors.')
    print('\nFeature Summary Statistics (Grouped by Class):')
    display_cols = ['ela_mean', 'ela_std', 'noise_cv', 'laplacian_var', 'dct_high_freq_ratio']
    print(df_feats.groupby('class_name')[display_cols].mean())
else:
    print('Features CSV not found. Run src/ml/train_classical.py first.')

## 5. Machine Learning Classifier Inference Test

In [ ]:
classifier = VeritasClassifier()
print(f'Classifier Status: Trained = {classifier.is_trained}')

pred_auth = classifier.predict(img_auth)
print('\nAuthentic Sample Prediction:')
print(f'Verdict: {pred_auth.prediction} | Manipulated Probability: {pred_auth.manipulated_probability}')

pred_spliced = classifier.predict(img_spliced)
print('\nSpliced Sample Prediction:')
print(f'Verdict: {pred_spliced.prediction} | Manipulated Probability: {pred_spliced.manipulated_probability}')